In [66]:
import numpy as np
import matplotlib.pyplot as plt
from amuse.lab import (Particles, ph4, units, constants)
from amuse.units import nbody_system
from amuse.community.huayno import Huayno
from amuse.community.ph4 import Ph4
from amuse.plot import scatter

## actually, no....
we dont really care about the negative energy cases, we just want to study the encounter of a black hole with another one/another binary system. it makes much more sense to sweep $v_{\infty}$ and $b$ as the parameters. Then:
#### $E = \frac{1}{2}\mu v_{\infty}^2$ 
#### $L = \mu b v_{\infty}$  
so the grid can be constructed using $b$ and $v_{\infty}$ 

In [38]:
#from the top, using v_inf and the impact parameter

def get_particleset(m1, m2, v_inf, b, r_init):
    """given masses of the 2 objects (m2 being the 'incoming' object), velocity 'at infinity', the collision 
    impact parameter and the initial seperation, gives corresponding particle set"""

    #velocities
    v_init = np.sqrt((v_inf**2)+(2*constants.G.number*(m1+m2)/r_init))
    v_init_perp = b*v_inf/r_init
    v_init_rad = -np.sqrt(v_init**2 - v_init_perp**2)

    #define the binary particle set
    two_black_holes = Particles(2)

    # declare the first BH
    bh1 = two_black_holes[0]
    bh1.mass = m1 | units.kg
    bh1.position = (0, 0, 0) | units.m
    bh1.velocity = (0, 0, 0) | units.ms

    # declare the second BH
    bh2 = two_black_holes[1]
    bh2.mass = m2 | units.kg
    bh2.position = (r_init, 0, 0) | units.m
    bh2.velocity = (v_init_rad, -v_init_perp, 0) | units.ms

    #print(f'before COM: {two_black_holes}')
    two_black_holes.move_to_center()

    return two_black_holes

In [53]:
#evolving the 2 body system

def integrate_2_body_huayno(particleset, time_step=0.01 | units.day, time_end=0.1 | units.day):
    
    convert_nbody = nbody_system.nbody_to_si(
        particleset.mass.sum(),
        particleset[1].position.length(),
    )

    gravity = Huayno(convert_nbody)
    particles_in_code = gravity.particles.add_particles(particleset)

    while gravity.model_time < time_end:
        gravity.evolve_model(gravity.model_time + (time_step))
        particles_in_code.new_channel_to(particleset).copy()
        particleset.savepoint(timestamp=gravity.model_time)

    gravity.stop()

    return particleset

In [ ]:
m1 = 1.98e30
m2 = 1.98e30
v_inf = 100000
b = 1e4
r_init = 50*b

particleset = get_particleset(m1, m2, v_inf, b, r_init)
# scatter(particleset.x, particleset.y, c='blue', label='before')
# print(particleset)

# integrated = integrate_2_body_huayno(particleset)
# scatter(integrated.x, integrated.y, c='magenta', label='after')
# #print(integrated)

# plt.legend()
# plt.show()



# converter = nbody_system.nbody_to_si(particleset.mass.sum(), particleset[0].position.length())
# gravity = Ph4(converter)
# gravity.parameters.timestep_parameter = 0.01
# gravity.particles.add_particles(particleset)
# print(f'gravity.particles: {gravity.particles}')
# print(f'bodies (bh1 and bh2): {particleset}')

# channel_from_gravity = gravity.particles.new_channel_to(particleset)
# channel_from_gravity.copy()
# print(f'updated bodies (star and bh1 and bh2): {particleset}')

# Porb = 2 | units.yr
# time = 0 | units.yr
# dt = Porb / 100
# index = 0
# while gravity.model_time < Porb:
#     time += dt
#     gravity.evolve_model(time)
#     channel_from_gravity.copy()


    
#     print(f"time={time.in_(units.yr)}")

#     plt.xlim(-500, 500)
#     plt.ylim(-500, 500)
#     scatter(particleset.x, particleset.y)
#     index += 1        
#     filename = f"snapshot_i{index:04}.jpg"
#     plt.savefig( filename)
#     plt.close()
# gravity.stop()





if __name__ == "__main__":
    particles = particleset

    
    converter = nbody_system.nbody_to_si(particles.mass.sum(), particles.position.length())
    gravity = Ph4(converter)
    gravity.parameters.timestep_parameter = 0.01
    gravity.particles.add_particles(particles)

    channel_from_gravity = gravity.particles.new_channel_to(particles)
    channel_from_gravity.copy()

    #Ek0 = star_both_blackholes.kinetic_energy()
    #Ep0 = star_both_blackholes.potential_energy()
    #Etot0 = Ek0 + Ep0

    time = 0 | units.yr
    dt = 30 | units.day
    index = 0
    while gravity.model_time < 1|units.yr:
        time += dt
        gravity.evolve_model(time)
        channel_from_gravity.copy()

        #Ek = star_both_blackholes.kinetic_energy()
        #Ep = star_both_blackholes.potential_energy()
        #Etot = Ek + Ep
        
        print(f"time={time.in_(units.yr)}") #, dE = {(Etot-Etot0)/Etot0}")

        scatter(particles.x, particles.y, s=1)
        index += 1        
        #filename = f"snapshot_i{index:04}.jpg"
        #plt.savefig( filename)
        #plt.close()
    gravity.stop()
    #plt.xscale('log')
    plt.xlim(-1e27, 1e27)
    plt.ylim(-1e27, 1e27)
    plt.show()
    print(particles)

gravity.particles:                  key         mass  potential_in_code       radius     timestep           vx           vy           vz            x            y            z
                   -           kg  m**2 * s**-2            m            s    m * s**-1    m * s**-1    m * s**-1            m            m            m
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========
14071768804316224036    1.980e+30   -2.643e+14   -2.500e+05    3.754e-06    1.626e+07    1.000e+03    0.000e+00   -2.500e+05    0.000e+00    0.000e+00
 1883026153306276924    1.980e+30   -2.643e+14   -2.500e+05    3.754e-06   -1.626e+07   -1.000e+03    0.000e+00    2.500e+05    0.000e+00    0.000e+00
====================  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========  ===========
bodies (bh1 and bh2):                  key         mass           vx

KeyboardInterrupt: 